# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# from langchain_community.document_loaders import WebBaseLoader

# url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"
# loader = WebBaseLoader(url)
# docs = loader.load()

# document_text = ""
# for doc in docs:
#     document_text += doc.page_content + "\n"

# len(document_text)
# print(document_text)

In [7]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "documents\\managing_oneself.pdf" 
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

document_text = ""
for i, page in enumerate(docs, start=1):
    content = (page.page_content or "").strip()
    if content:
        document_text += f"\n\n--- Page {i} ---\n{content}"


## Generation Task

Using the OpenAI SDK, please create a **structured ouput** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [8]:
from openai import OpenAI

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="dummy",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

TONE = "Formal Academic Writing"
MODEL = "gpt-4o-mini"

system_prompt = f"""
You are a precise academic summarizer.
Return ONLY valid JSON matching the schema below.
Write Relevance and Summary in tone: {TONE}
Ground every sentence strictly in the provided document.
Do not invent facts, numbers, names or concepts.
""".strip()

user_prompt = f"""
DOCUMENT:
{document_text}

Extract metadata and write summary following the instructions.
""".strip()

schema = ArticleSummary.model_json_schema()
# optional: enforce additionalProperties: false recursively (common fix)
def enforce_strict(d):
    if isinstance(d, dict):
        if d.get("type") == "object" and "additionalProperties" not in d:
            d["additionalProperties"] = False
        for v in d.values():
            enforce_strict(v)
    elif isinstance(d, list):
        for x in d: enforce_strict(x)
enforce_strict(schema)

response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "summary",
            "schema": schema,
            "strict": True
        }
    }
)

data = json.loads(response.output_text)
data["InputTokens"] = response.usage.input_tokens
data["OutputTokens"] = response.usage.output_tokens
data["Tone"] = data.get("Tone", TONE)

summary_v1 = ArticleSummary.model_validate(data)
print(summary_v1.model_dump_json(indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article emphasizes the necessity for individuals to take control of their own careers in the context of a rapidly evolving knowledge economy, where traditional corporate structures no longer guarantee career development. Drucker advocates for self-awareness, specifically in recognizing one’s strengths, values, and preferred work styles, as essential to achieving sustained performance and fulfillment in one's career. It underscores the shift in responsibility from companies to individuals, highlighting the importance of personal accountability in career management.",
  "Summary": "In the contemporary landscape marked by significant opportunity for knowledge workers, personal accountability and self-management emerge as critical. Drucker posits that workers must act as their own chief executive officers, requiring an adept understanding of their own strengths, weaknesses, work preferences, and values. E

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import DeepEvalBaseLLM

class GatewayLLM(DeepEvalBaseLLM):
    def __init__(self, model="gpt-4o"):
        self.client = OpenAI(
            base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
            api_key="any value",                     # ← must match what worked before
            default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
        )
        self.model = model

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,                        # optional but recommended for evaluation
            max_tokens=2000
        )
        return resp.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return f"gateway-{self.model}"

judge = GatewayLLM("gpt-4o")

test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_v1.Summary
)

# Summarization metric with custom questions
summ_metric = SummarizationMetric(
    model=judge,
    assessment_questions=[
        "Does the summary capture the main thesis?",
        "Are the key supporting arguments included?",
        "Is anything important missing?",
        "Does it contain unsupported claims?",
        "Is the logical order of ideas preserved?"
    ],
    include_reason=True
)

# G-Eval examples
coherence = GEval(
    name="Coherence",
    model=judge,
    evaluation_steps=[
        "Logical flow and structure",
        "Smooth transitions",
        "No contradictions",
        "Easy to follow",
        "Sensible ordering of points"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

tonality = GEval(
    name="Tonality",
    model=judge,
    evaluation_steps=[
        f"Consistent {TONE} throughout",
        "No informal/slang language",
        "Vocabulary matches register",
        "Tone supports clarity",
        "Uniform style end-to-end"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

safety = GEval(
    name="Safety",
    model=judge,
    evaluation_steps=[
        "No toxic or harmful content",
        "No biased language",
        "No dangerous instructions",
        "No hallucinated sensitive facts",
        "Suitable for professional audience"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

# Run evaluations
for metric in [summ_metric, coherence, tonality, safety]:
    metric.measure(test_case)

results = {}
for m in [summ_metric, coherence, tonality, safety]:
    results[f"{m.name}Score"]  = m.score
    results[f"{m.name}Reason"] = m.reason

print(json.dumps(results, indent=2))

TypeError: Invalid type for url.  Expected str or httpx.URL, got <class 'ellipsis'>: Ellipsis

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.

In [6]:
import json

improvement_instructions = f"""
You previously wrote a summary that was evaluated with the following feedback:

Summarization feedback:
- Score: {eval_result.get("SummarizationScore")}
- Reason: {eval_result.get("SummarizationReason")}

Coherence feedback:
- Score: {eval_result.get("CoherenceScore")}
- Reason: {eval_result.get("CoherenceReason")}

Tonality feedback:
- Score: {eval_result.get("TonalityScore")}
- Reason: {eval_result.get("TonalityReason")}

Safety feedback:
- Score: {eval_result.get("SafetyScore")}
- Reason: {eval_result.get("SafetyReason")}


Now produce an improved version with these hard constraints:
1) Do NOT add facts, numbers, or claims that are not explicitly supported by the provided document context.
2) If a detail (e.g., investment amounts, percentages, named concepts) is not found in the context, omit it or hedge (e.g., "the document suggests...") without inventing values.
3) Resolve the evaluation issues specifically mentioned above (especially hallucinations / contradictions).
4) Maintain the distinctive tone: {TONE}.
5) Keep Relevance ≤ 1 paragraph. Keep Summary ≤ 1000 tokens.
6) Output must be ONLY valid JSON that matches the given schema, with no extra keys.
""".strip()


schema_v2 = ArticleSummary.model_json_schema()
schema_v2 = enforce_additional_properties_false(schema_v2)

developer_v2 = f"""
You are a document analyst. Return ONLY valid JSON matching the required schema.
Write the Summary and Relevance in the distinctive tone: {TONE}.

STRICT GROUNDING & STYLE:
- Use ONLY information explicitly present in the document context.
- Prefer extractive phrasing; paraphrase minimally and faithfully.
- If the document does not explicitly state a thesis sentence, start with: "The article argues that ..."
- Preserve the source’s ordering of ideas; do not reorder by theme unless the article does.
- Do NOT add examples, analogies, or background beyond the source.
- No rhetorical questions, idioms, or conversational asides.
- Use neutral, precise vocabulary typical of scholarly abstracts.
- Avoid second person; avoid normative judgments not present in the text.

SUMMARY STRUCTURE:
1) First sentence: a single clear thesis statement of the article.
2) Middle: 3–7 concise, logically ordered points that follow the source’s progression.
3) Final sentence (optional): only if the document itself provides a concluding claim.

CONSTRAINTS:
- Summary ≤ 1000 tokens.
- Relevance: ≤ one paragraph; explain why the article matters to an AI professional.
- If author is not stated, return "Unknown".
- Return ONLY JSON following the provided schema; no extra keys.

"""

user_v2 = f"""
DOCUMENT CONTEXT:
{document_text}

PREVIOUS SUMMARY (v1):
{summary_result.model_dump_json(indent=2)}

ENHANCEMENT TASK:
{improvement_instructions}

Rewrite the summary to resolve the feedback while strictly adhering to the constraints and structure.
If the previous summary contained details not supported by the document, omit them.
Return ONLY valid JSON per the schema; no extra keys.

"""

response_v2 = client.responses.create(
    model=MODEL,  
    input=[
        {"role": "developer", "content": developer_v2},
        {"role": "user", "content": user_v2},
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "article_summary_v2",
            "schema": schema_v2,
            "strict": True,
        }
    },
)

usage_v2 = getattr(response_v2, "usage", None)
in_tok_v2 = int(getattr(usage_v2, "input_tokens", 0) or 0)
out_tok_v2 = int(getattr(usage_v2, "output_tokens", 0) or 0)

summary_json_v2 = json.loads(response_v2.output_text)
summary_json_v2["InputTokens"] = in_tok_v2
summary_json_v2["OutputTokens"] = out_tok_v2
summary_json_v2["Tone"] = summary_json_v2.get("Tone") or TONE

summary_result_v2 = ArticleSummary.model_validate(summary_json_v2)

print("=== v2 Summary ===")
print(summary_result_v2.model_dump_json(indent=2))

test_case_v2 = LLMTestCase(
    input=document_text,
    actual_output=summary_result_v2.Summary
)

summ_metric.measure(test_case_v2)
coherence_metric.measure(test_case_v2)
tonality_metric.measure(test_case_v2)
safety_metric.measure(test_case_v2)

eval_result_v2 = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

print("=== v2 Evaluation ===")
print(json.dumps(eval_result_v2, indent=2, ensure_ascii=False))


Output()

=== v2 Summary ===
{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article is significant for AI professionals as it underscores the importance of self-awareness in personal and professional development, which is essential in a rapidly changing technological landscape where collaboration and understanding individual capabilities are paramount.",
  "Summary": "The article argues that success in the knowledge economy necessitates self-knowledge, which compels individuals to act as their own chief executive officers. Drucker posits that individuals must take ownership of their careers, recognizing that organizations can no longer be relied upon to manage knowledge workers' paths. A thorough understanding of one's strengths, weaknesses, learning styles, and values is essential for effective self-management. Feedback analysis serves as a critical tool for individuals to identify their strengths and areas needing development. Furthermore, understanding h

Output()

Output()

Output()

=== v2 Evaluation ===
{
  "SummarizationScore": 0.7142857142857143,
  "SummarizationReason": "The score is 0.71 because the summary includes extra information not present in the original text, such as weaknesses, learning styles, and anticipating changes. Additionally, the summary leaves out the ability to answer some specific questions that the original text can address. Despite these issues, there is no contradictory information present.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The response follows a clear logical flow and cohesive structure, detailing key points like self-management, feedback analysis, and alignment of values. Transitions between ideas are smooth, enhancing readability and understanding. There are no contradictions or confusing references, and the ordering of points is logical, culminating in the emphasis on career management. A minor shortcoming is the lack of explicit reference to the term 'knowledge economy' throughout the article, which slightly affects 


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
